# 열연 압연 하중 데이터 불러오기

원본: `../main/raw_data/metals.xlsx` (시트 `All Grades Data`)

엑셀이 Strict OOXML 형식이라 `pd.read_excel`이 시트를 읽지 못합니다. XML을 직접 파싱합니다.

In [ ]:
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd

NS = "{http://purl.oclc.org/ooxml/spreadsheetml/main}"
XLSX = Path("../main/raw_data/metals.xlsx")


def _col_index(ref):
    n = 0
    for ch in re.match(r"[A-Z]+", ref).group():
        n = n * 26 + ord(ch) - 64
    return n - 1


def load_metals(path=XLSX, sheet="All Grades Data", n_cols=10):
    """Strict OOXML 엑셀의 한 시트를 DataFrame으로 읽는다."""
    with zipfile.ZipFile(path) as z:
        wb = z.read("xl/workbook.xml").decode("utf-8")
        rels = z.read("xl/_rels/workbook.xml.rels").decode("utf-8")
        rid = re.search(r'<sheet name="%s"[^>]*r:id="([^"]+)"' % re.escape(sheet), wb).group(1)
        target = re.search(r'<Relationship [^>]*Id="%s"[^>]*Target="([^"]+)"' % rid, rels)
        target = target.group(1) if target else re.search(r'Target="([^"]+)"[^>]*Id="%s"' % rid, rels).group(1)
        target = target.lstrip("/")
        sheet_path = target if target.startswith("xl/") else "xl/" + target

        shared = [
            "".join(t.text or "" for t in si.iter(NS + "t"))
            for si in ET.fromstring(z.read("xl/sharedStrings.xml")).iter(NS + "si")
        ]

        rows = []
        with z.open(sheet_path) as f:
            for _, el in ET.iterparse(f):
                if el.tag != NS + "row":
                    continue
                row = [None] * n_cols
                for c in el.findall(NS + "c"):
                    i = _col_index(c.get("r"))
                    if i >= n_cols:
                        continue
                    v = c.find(NS + "v")
                    if v is None:
                        continue
                    row[i] = shared[int(v.text)] if c.get("t") == "s" else v.text
                rows.append(row)
                el.clear()

    df = pd.DataFrame(rows[1:], columns=rows[0])
    num_cols = [c for c in df.columns if c not in ("Grade", "Caster ID", "Stand ID")]
    df[num_cols] = df[num_cols].apply(pd.to_numeric)
    df["Coil ID"] = df["Coil ID"].astype(int)
    return df


df = load_metals()
print(df.shape)
df.head()